# 课堂教学案例：西财新闻怎么拉下来

> 给谁：W2 第 2 节「拉一次数据」跟做；装不上 opencode 的用 Trae 或自己敲。  
> 数据：[西财要闻](https://www.swufe.edu.cn/index/xcyw.htm)  
> 工具：`requests` 下载 HTML，`BeautifulSoup` 从 HTML 里取出标题、日期、链接、正文。  
> 当堂：跑通「列表首页 + 1 篇正文」。翻完全站、下完全文留课后（必须间隔请求，不要把学校网站打挂）。  
> 完整脚本：同目录 `swufe-news.py`（默认就是当堂模式）。


## 0. 先想清楚：分析西财近些年新闻，该如何做

不要一上来就写爬虫。先问四件事（对应 `data-taxonomy.md`）：

| 问 | 这份新闻大概是 |
|---|---|
| 从哪来 | **客观**（学校发过的稿），不是合成 |
| 谁能看 | **公开**（官网，不用登录） |
| 花不花钱 | **免费** |
| 怎么拿 | **网页通道**（没有现成 API，HTML 里就有） |
| 长什么样 | 列表是半结构化；正文是 **非结构化文本** |
| 能不能进产品 | 学习、分析可以；转载原文要看学校规定，默认只存自己用 |

分析路径也先写在纸上，再动手：

1. **列表**：每条新闻的日期、标题、链接
2. **正文**：点进去拿到完整段落
3. **落盘**：csv / jsonl，方便以后按年统计、做词频、选题
4. **分析**（本周不必做完）：近年发稿量、主题词、哪些学院常出现——那是有了数据之后的事

本页只做到第 3 步。网页采集的系统讲法见 W4 的 `knowledge/phase1-1-4b-web-data-collection.md`。

环境（在项目 `.venv` 里）：

```text
python -m pip install requests beautifulsoup4 pandas
```


## 1. 两个工具各干什么

打开网页，浏览器其实做了两件事：向服务器要一份 HTML 文件，再把文件画成你看见的页面。

- **`requests`**：替你「要文件」。给一个网址，返回状态码和文本。它**不理解**标题在哪。
- **`BeautifulSoup`**：替你「在这份 HTML 里找标签」。比如所有 `h3`、所有带 `class="publine"` 的链接。

口诀：**requests 负责拿，BeautifulSoup 负责拆。** 先拿后拆。编码不对会乱码，西财这站用 UTF-8：`resp.encoding = "utf-8"`。

礼貌：学校官网 `robots.txt` 当前是 404，仍要当客人——加 `User-Agent`、设 `timeout`、翻页时 `time.sleep`，课堂不要开多线程狂刷。


## 2. 浅：先把首页 HTML 拿到手


In [6]:
import requests

URL = "https://www.swufe.edu.cn/index/xcyw.htm"
headers = {"User-Agent": "SWUFE-DA-class/0.1 (teaching; polite crawl)"}

resp = requests.get(URL, headers=headers, timeout=15)
resp.encoding = "utf-8"
print(resp.status_code)      # 200 才算拿到
print(resp.text[:400])       # 先看是不是 HTML，不要假设


200
﻿<!DOCTYPE html>
<html lang="zh-CN">

<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=0,minimal-ui:ios">
    <meta http-equiv="X-UA-Compatible" content="ie=edge">
    <title>西财要闻-西南财经大学</title><meta name="pageType" content="2">
<meta name="pageTitle" content="西财要闻">
<META Name="keywords" Content="


`200` 再往下走。不是 200：把状态码记进可得性初验，换源或下课再试。


## 3. 拆列表：日期、标题、链接

用浏览器「检查」会看到每条新闻是：

```html
<a href="../info/1048/50992.htm" class="publine">
  <div class="date"><span>2026-09-12</span></div>
  <div class="info"><h3>标题</h3><p>摘要…</p></div>
</a>
```

相对路径要拼成绝对地址，用 `urljoin`，不要手搓。首页链接是 `../info/...`，翻到第 2 页会变成 `../../info/...`，层数不同，**必须**用当前列表页的完整 URL 去 join。


In [8]:
from urllib.parse import urljoin
from bs4 import BeautifulSoup

soup = BeautifulSoup(resp.text, "html.parser")
rows = []
for a in soup.select("a.publine"):
    href = a.get("href")
    if not href:
        continue
    title = a.select_one("h3")
    date = a.select_one(".date span")
    rows.append({
        "date": date.get_text(strip=True) if date else "",
        "title": title.get_text(strip=True) if title else "",
        "url": urljoin(URL, href),
    })

print(len(rows))
print(rows[0])


10
{'date': '2026-06-24', 'title': '教育部党组书记、部长怀进鹏调研西南财经大学', 'url': 'https://www.swufe.edu.cn/info/1048/49802.htm'}


当堂过关：能打印出大约 10 条（首页一页的条数），第一条有日期和标题。导航栏里也有链接，所以**只选 `a.publine`**，不要 `soup.select("a")` 一把抓。


## 4. 再深一点：一篇正文

列表页只有摘要。正文在 `https://www.swufe.edu.cn/info/1048/xxxxx.htm`，正文容器是 `.v_news_content`。


In [3]:
import time

def fetch_article(url):
    r = requests.get(url, headers=headers, timeout=15)
    r.encoding = "utf-8"
    s = BeautifulSoup(r.text, "html.parser")
    h = s.select_one(".art-tit h3")
    body = s.select_one(".v_news_content")
    return {
        "url": url,
        "title": h.get_text(strip=True) if h else "",
        "text": body.get_text("\n", strip=True) if body else "",
    }

sample = fetch_article(rows[0]["url"])
print(sample["title"])
print(sample["text"][:200])


教育部党组书记、部长怀进鹏调研西南财经大学
6月22日，教育部党组书记、部长怀进鹏一行到西南财经大学调研，了解学校服务国家重大战略和区域经济社会发展、服务构建中国特色自主知识体系、推动教育科技人才一体发展、培养特色学科人才等情况，强调要深入学习贯彻习近平党建思想，扎实开展树立和践行正确政绩观学习教育。四川省人民政府副省长胡云，学校党委书记赵建军，党委副书记、校长李永强陪同调研，全体在校校领导参加调研座谈会。
怀进鹏前往校史馆，了解学校历史沿


当堂只拉 **1 篇**。确认有完整段落，再谈翻页。


## 课后练习题

- 用opencode写代码，存储为py代码
- 下载西财所有要闻